# 📚 Sigma Books：传奇 AI 书店助手

> **基于 Gradio 构建 | 毒舌与品味并存**

## 练习目标

做一个带 **人设（persona）** 的书店聊天助手：

- 用 **DeepSeek**（OpenAI 兼容 SDK）当后端模型
- 用 **system prompt** 固定「毒舌策展人」口吻与禁言情等规则
- 用 **Gradio `ChatInterface`** 做流式（streaming）对话界面

这对应课程里常见的三件套：**密钥进环境变量 → messages（system/user/history）→ UI 流式展示**。

## 怎么跑

1. 准备 `.env`：至少有 `DEEPSEEK_API_KEY`
2. 从上到下依次运行单元格
3. 在弹出的 Gradio 窗口里和「书店策展人」聊天

---

欢迎来到 **Sigma Books** 的数字店面。这不是那种过分客气的普通聊天机器人——而是一位高能、机智、略带评判的书店策展人，帮你找到完美读物（顺便友好地吐槽你的纠结）。

### 功能

* **专家策展：** 从 **动作**、**历史** 到 **政治**、**体育**——只要够传奇，我们都有。
* **「禁言情」专区：** 我们只卖有料的书，不卖煽情续集。敢要言情，后果自负。
* **高价值推荐：** 我们会告诉你为什么《牧羊少年奇幻之旅》是你花过最值的 **N10,000**。
* **互动式幽默：** 用 **Gradio** 打造流畅、有趣的聊天体验。

> *"我们有你正在想的那本书。说出书名，别的不用说。"*

---

*享受传奇书籍的世界吧！*


In [ ]:
# ========== 导入：后面要用的库一次搬进来 ==========

# 标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# dotenv：从 .env 文件加载密钥到进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# OpenAI 官方 SDK：这里用来对接「兼容 OpenAI 协议」的 DeepSeek 接口
from openai import OpenAI
# Gradio：快速搭 Web 聊天 UI（ChatInterface）
import gradio as gr


In [ ]:
# ========== 环境变量 + DeepSeek 服务地址 ==========

# override=True：.env 里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境读取 DeepSeek API Key（名字必须是 DEEPSEEK_API_KEY，与 .env 一致）
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
# DeepSeek 的 OpenAI 兼容 API 基址（URL 字符串勿改）
deepseek_url = "https://api.deepseek.com" 


In [ ]:
# ========== 客户端 + 模型名 ==========

# 用 OpenAI 客户端类指向 DeepSeek：base_url 换服务，api_key 用 DeepSeek 的密钥
deepseek = OpenAI(base_url=deepseek_url, api_key=deepseek_api_key)
# 模型 id：deepseek-chat（字符串影响实际调用的模型，勿改）
MODEL = 'deepseek-chat'


In [ ]:
# ========== 系统提示词（system prompt）：锁定人设与销售规则 ==========
# 下面整段英文是发给模型的「行为说明书」：人设、品类、禁言情、价位话术、开场白等
# 教学注释可写中文，但 prompt 字面量本身不要翻译/改写，否则回答风格会变

system_prompt = "You are the funny, sharp-witted curator of Sigma Books, the world of legendary literature." \
" Your goal is to sell books with a mix of dry humor, high-energy marketing, and playful judgment." \
" You treat our collection—Action, Fiction, Education, History, Politics, Sports, and Comedy—like holy relics." \
" HOWEVER, we strictly DO NOT stock romance; if asked for it, respond with witty horror or mock disappointment." \
" Your tone is irreverent and persuasive—never announce jokes with 'here is a joke,' just be naturally funny." \
" Use the N8,000 - N10,000 price point for top-tier recommendations like 'The Alchemist,' telling customers it would be worth it (in your own way)." \
" Start conversations with: 'Welcome to Sigma Books. We have the book you are thinking of. Just say the name and say no more.', or something better of your own" \
"choice. Just be Super funny, and do not be rude in the slightest of ways" \
" Listen to their needs, roast their indecision if necessary, and close the sale with confidence."


In [ ]:
# ========== 聊天回调：拼 messages + 流式 yield ==========

def chatt(message, history):
    # Gradio messages 格式 → 只保留 role/content，方便塞进 Chat Completions
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 顺序：system（人设）+ 历史轮次 + 当前用户输入
    messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user", "content":message}]
    # stream=True：服务端边生成边推送，客户端可逐步展示（打字机效果）
    stream = deepseek.chat.completions.create(model=MODEL, messages=messages, stream=True)

    # 累积已生成文本；每来一块就 yield 一次「到目前为止的完整回复」
    response = ""
    for chunk in stream:
        # delta.content 可能是 None（例如结束块），用 or '' 避免拼接报错
        response += chunk.choices[0].delta.content or ''
        # Gradio 生成器协议：不断 yield 更新 UI
        yield response


In [ ]:
# ========== 启动 Gradio 聊天界面 ==========

# fn=chatt：用户每发一条就调用上面的流式回调
# type="messages"：历史用 role/content 消息列表（与 OpenAI messages 风格对齐）
gr.ChatInterface(fn=chatt, type="messages").launch()
